# Deploy LangGraph Agent to Databricks

This notebook provides a simplified, self-contained way to deploy a LangGraph multi-agent system to Databricks.

**Key Features:**
- Hardcoded configurations for simplicity
- Creates and logs the agent to MLflow
- Registers the model in Unity Catalog
- Deploys to Databricks Model Serving
- Includes deployment monitoring and testing

**Prerequisites:**
- Running in a Databricks workspace
- Access to Unity Catalog
- Snowflake secrets configured in Databricks (for SQL workflow)
- Foundation model endpoint available

## 1. Install Dependencies and Import Libraries

In [3]:
# Install required packages (if needed)
# %pip install mlflow databricks-sdk langgraph langchain-core langchain-community

import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import mlflow
from databricks.sdk import WorkspaceClient
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint

# Set MLflow tracking URI to Databricks
mlflow.set_tracking_uri("databricks")

print("✓ Dependencies imported successfully")

✓ Dependencies imported successfully


## 2. Validate Setup

Check that all required files and dependencies are available.

In [4]:
# Validation Cell - Check workspace structure and dependencies
print("=" * 70)
print("WORKSPACE VALIDATION")
print("=" * 70)

# Check workspace structure
project_root = Path.cwd()
print(f"\nProject root: {project_root}")

required_paths = [
    project_root / "sql_workflow",
    project_root / "sql_workflow_databricks.py",
    project_root / "sql_workflow" / "__init__.py",
    project_root / "sql_workflow" / "graph_builder.py",
]

print("\n✓ Checking required files:")
all_exist = True
for path in required_paths:
    exists = path.exists()
    status = "✓" if exists else "✗"
    print(f"  {status} {path.name}")
    all_exist = all_exist and exists

if not all_exist:
    raise FileNotFoundError("Missing required files! Check workspace structure above.")

# Check foundation model access
print("\n✓ Checking foundation model access:")
try:
    from databricks.sdk import WorkspaceClient
    ws = WorkspaceClient()
    endpoint = ws.serving_endpoints.get("databricks-meta-llama-3-1-70b-instruct")
    print(f"  ✓ Foundation model accessible: {endpoint.name}")
except Exception as e:
    print(f"  ⚠ Model access check failed: {e}")

print("\n" + "=" * 70)
print("✓ VALIDATION COMPLETE - Ready to proceed!")
print("=" * 70)

WORKSPACE VALIDATION

Project root: /Users/pragati.gupta/Projects/LanggraphToDatabricks

✓ Checking required files:
  ✓ sql_workflow
  ✓ sql_workflow_databricks.py
  ✓ __init__.py
  ✓ graph_builder.py

✓ Checking foundation model access:
  ⚠ Model access check failed: Endpoint with name 'databricks-meta-llama-3-1-70b-instruct' does not exist.

✓ VALIDATION COMPLETE - Ready to proceed!


## 2. Configuration (Hardcoded)

Update these values according to your Databricks environment:

In [5]:
# =============================================================================
# CONFIGURATION - Update these values for your environment
# =============================================================================

# Catalog and Schema Configuration
CATALOG_NAME = "main"  # Your Unity Catalog name - UPDATE THIS
GOLD_SCHEMA = "default"  # Your schema name - UPDATE THIS
RESOURCE_PREFIX = ""  # Optional prefix for resources
USER_NAME = "pragati.gupta"  # Current user (extracted from workspace path)

# Model Configuration
FOUNDATION_MODEL_NAME = "databricks-meta-llama-3-1-70b-instruct"  # Foundation model endpoint
MULTIAGENT_MODEL_NAME = f"{CATALOG_NAME}.{GOLD_SCHEMA}.sql_workflow_agent"  # Registered model name
ENDPOINT_NAME = "sql-workflow-agent-endpoint"  # Serving endpoint name (no underscores)

# Experiment Configuration  
EXPERIMENT_PATH = f"/Users/{USER_NAME}/sql-workflow-experiments"  # MLflow experiment path

print("=" * 70)
print("CONFIGURATION LOADED")
print("=" * 70)
print(f"Model: {MULTIAGENT_MODEL_NAME}")
print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Experiment: {EXPERIMENT_PATH}")
print(f"Foundation Model: {FOUNDATION_MODEL_NAME}")
print(f"")
print(f"NOTE: Using MOCK DATA - No database connection required!")
print("=" * 70)

CONFIGURATION LOADED
Model: main.default.sql_workflow_agent
Endpoint: sql-workflow-agent-endpoint
Experiment: /Users/pragati.gupta/sql-workflow-experiments
Foundation Model: databricks-meta-llama-3-1-70b-instruct

NOTE: Using MOCK DATA - No database connection required!


## 3. Create Agent Script

This cell creates the agent script that will be logged and deployed. The agent should be defined in a separate Python file that MLflow can package.

In [6]:
# Create the agent script file
# This references the sql_workflow_databricks.py file in your workspace

agent_script_content = f'''
import sys
from pathlib import Path

# Add the parent directory to the path to find sql_workflow package
parent_dir = Path(__file__).parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

# Import the SQL workflow orchestrator from your workspace
from sql_workflow_databricks import SQLWorkflowOrchestrator

# Initialize the SQL Workflow Orchestrator with the foundation model
orchestrator = SQLWorkflowOrchestrator(model="{FOUNDATION_MODEL_NAME}")

# Set as MLflow agent model
import mlflow
mlflow.models.set_model(orchestrator)
'''

# Write the agent script to a temporary file
agent_file_path = Path("/tmp/sql_workflow_agent.py")
with open(agent_file_path, "w") as f:
    f.write(agent_script_content)

print("=" * 70)
print("✓ Agent script created successfully")
print("=" * 70)
print(f"Script location: {agent_file_path}")
print(f"Foundation model: {FOUNDATION_MODEL_NAME}")
print("=" * 70)

✓ Agent script created successfully
Script location: /tmp/sql_workflow_agent.py
Foundation model: databricks-meta-llama-3-1-70b-instruct


## 4. Log Model to MLflow

Log the agent to MLflow with all dependencies and resources.

In [ ]:
# Create or get MLflow experiment
experiment = mlflow.set_experiment(EXPERIMENT_PATH)
print(f"✓ Using experiment: {experiment.name} (ID: {experiment.experiment_id})")

# Prepare resources that the agent needs access to
resources = [
    DatabricksServingEndpoint(endpoint_name=FOUNDATION_MODEL_NAME),
    DatabricksFunction(function_name="system.ai.python_exec"),
]

print(f"✓ Configured {len(resources)} resources")

# Prepare sample request for input example
sample_request = {
    "input": [
        {
            "role": "user",
            "content": "Which customer placed the maximum orders in the last month?",
        }
    ]
}

# Create run name with timestamp
job_run_ts = datetime.now(tz=timezone.utc).strftime("%Y%m%d-%H%M%S")
run_name = f"sql-workflow-run-{job_run_ts}"

print(f"✓ Starting MLflow run: {run_name}")

In [ ]:
# Log the model to MLflow with correct code paths
with mlflow.start_run(run_name=run_name, log_system_metrics=True) as run:
    # Get the project root (where this notebook is located)
    project_root = Path.cwd()
    
    print(f"Project root: {project_root}")
    print(f"Code paths:")
    print(f"  - {project_root / 'sql_workflow'}")
    print(f"  - {project_root / 'sql_workflow_databricks.py'}")
    
    # Define pip requirements explicitly (no Snowflake dependencies needed)
    pip_requirements = [
        "mlflow>=2.9.0",
        "langchain-core>=0.1.0",
        "langchain-community>=0.0.1",
        "langgraph>=0.0.40",
        "databricks-sdk>=0.18.0",
        "databricks-langchain>=0.1.0",
        "openai>=1.0.0",
        "backoff>=2.2.0",
        "pydantic>=2.0.0",
    ]
    
    # Log model with all necessary code and dependencies
    model_info = mlflow.pyfunc.log_model(
        python_model=str(agent_file_path),
        code_paths=[
            str(project_root / "sql_workflow"),  # Main workflow package
            str(project_root / "sql_workflow_databricks.py"),  # Orchestrator module
        ],
        pip_requirements=pip_requirements,
        input_example=sample_request,
        resources=resources,
        artifact_path="model",
    )
    
    print("\n" + "=" * 70)
    print("✓ MODEL LOGGED SUCCESSFULLY")
    print("=" * 70)
    print(f"Run ID: {run.info.run_id}")
    print(f"Model URI: {model_info.model_uri}")
    print("=" * 70)
    
    # Store for next steps
    RUN_ID = run.info.run_id
    MODEL_URI = model_info.model_uri

## 5. Test Model Locally (Optional)

Test the model locally before registering and deploying.

In [ ]:
# Optional: Test the model locally
try:
    print("Testing model locally...")
    agent = mlflow.pyfunc.load_model(MODEL_URI)
    sample_response = agent.predict(sample_request)
    
    print("✓ Model test successful")
    print(f"  Request: {sample_request['input'][0]['content']}")
    print(f"  Response preview: {str(sample_response)[:200]}...")
    
except Exception as e:
    print(f"⚠ Local test failed (this may be OK): {e}")
    print("Continuing with registration...")

## 6. Register Model in Unity Catalog

In [ ]:
# Register the model in Unity Catalog
client = mlflow.MlflowClient()

# Create registered model if it doesn't exist
try:
    existing_models = [m.name for m in client.search_registered_models()]
    if MULTIAGENT_MODEL_NAME not in existing_models:
        client.create_registered_model(MULTIAGENT_MODEL_NAME)
        print(f"✓ Created registered model: {MULTIAGENT_MODEL_NAME}")
    else:
        print(f"✓ Using existing registered model: {MULTIAGENT_MODEL_NAME}")
except Exception as e:
    print(f"Model may already exist: {e}")

# Register the model version
model_version = client.create_model_version(
    name=MULTIAGENT_MODEL_NAME,
    source=MODEL_URI,
    run_id=RUN_ID
)

MODEL_VERSION = model_version.version

print(f"✓ Model registered as version {MODEL_VERSION}")
print(f"  - Name: {MULTIAGENT_MODEL_NAME}")
print(f"  - Version: {MODEL_VERSION}")

## 7. Deploy to Model Serving Endpoint

Deploy the registered model to a Databricks Model Serving endpoint with environment variables for Snowflake access.

In [ ]:
# Deploy the model to a serving endpoint (no environment variables needed for mock data)
print(f"Deploying model version {MODEL_VERSION} to endpoint '{ENDPOINT_NAME}'...")

try:
    agents.deploy(
        model_name=MULTIAGENT_MODEL_NAME,
        model_version=int(MODEL_VERSION),
        endpoint_name=ENDPOINT_NAME,
        # No environment variables needed - using mock data
    )
    
    print("✓ Deployment request submitted successfully")
    print(f"  - Endpoint: {ENDPOINT_NAME}")
    print(f"  - Model: {MULTIAGENT_MODEL_NAME} v{MODEL_VERSION}")
    print(f"  - Using MOCK DATA (no database connection required)")
    
except Exception as e:
    print(f"✗ Deployment failed: {e}")
    raise

## 8. Wait for Endpoint to be Ready

Monitor the deployment progress until the endpoint is ready to serve requests.

In [ ]:
def wait_for_endpoint_ready(endpoint_name: str, timeout_seconds: int = 1800, poll_interval: int = 30):
    """
    Poll serving endpoint until deployment is complete.
    
    Args:
        endpoint_name: Name of the serving endpoint
        timeout_seconds: Maximum time to wait (default: 1800 = 30 min)
        poll_interval: Time between status checks (default: 30s)
    """
    ws = WorkspaceClient()
    start_time = time.time()
    
    print(f"⏳ Waiting for endpoint '{endpoint_name}' to be ready...")
    print(f"   (timeout: {timeout_seconds}s, polling every {poll_interval}s)")
    
    while time.time() - start_time < timeout_seconds:
        try:
            endpoint = ws.serving_endpoints.get(endpoint_name)
            
            config_update = endpoint.state.config_update.name
            ready_state = endpoint.state.ready.name
            
            elapsed = int(time.time() - start_time)
            print(f"   [{elapsed}s] Status: config_update={config_update}, ready={ready_state}")
            
            # Check if update is complete
            if config_update == "NOT_UPDATING":
                if ready_state == "READY":
                    print(f"✓ Endpoint '{endpoint_name}' is READY and serving!")
                    return True
                else:
                    print(f"✗ Deployment failed with state: {ready_state}")
                    raise Exception(f"Deployment failed with state: {ready_state}")
            
            # Still updating, wait before next poll
            time.sleep(poll_interval)
            
        except Exception as e:
            if "does not exist" in str(e):
                print(f"✗ Endpoint '{endpoint_name}' not found")
                raise
            raise
    
    # Timeout reached
    elapsed = int(time.time() - start_time)
    raise TimeoutError(f"Endpoint deployment timeout after {elapsed}s")

# Wait for the endpoint to be ready
wait_for_endpoint_ready(ENDPOINT_NAME)

## 9. Test the Deployed Endpoint

Send a test query to verify the endpoint is working correctly.

In [ ]:
# Test the deployed endpoint
print(f"Testing endpoint: {ENDPOINT_NAME}")

# Create a test request
test_request = {
    "messages": [
        {
            "role": "user",
            "content": "What are the top 5 customers by order volume?"
        }
    ]
}

try:
    # Query the endpoint using Databricks SDK
    ws = WorkspaceClient()
    response = ws.serving_endpoints.query(
        name=ENDPOINT_NAME,
        inputs=test_request
    )
    
    print("✓ Endpoint test successful!")
    print(f"  Request: {test_request['messages'][0]['content']}")
    print(f"  Response: {response}")
    
except Exception as e:
    print(f"✗ Endpoint test failed: {e}")
    print("  The endpoint may still be initializing. Try again in a few minutes.")

## 10. Summary and Next Steps

**Deployment Complete!** 🎉

Your LangGraph agent has been successfully deployed to Databricks Model Serving.

**What was deployed:**
- Model: `{MULTIAGENT_MODEL_NAME}` version `{MODEL_VERSION}`
- Endpoint: `{ENDPOINT_NAME}`
- Foundation Model: `{FOUNDATION_MODEL_NAME}`

**Next steps:**
1. **Monitor the endpoint** in the Databricks Serving UI
2. **Set up Review App** for human feedback (optional)
3. **Configure auto-scaling** based on traffic patterns
4. **Set up alerts** for endpoint health monitoring
5. **Test with production queries** to validate performance

**To query the endpoint from code:**
```python
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
response = ws.serving_endpoints.query(
    name="{ENDPOINT_NAME}",
    inputs={"messages": [{"role": "user", "content": "Your question here"}]}
)
```

**Useful links:**
- Endpoint URL: `https://<workspace-url>/serving-endpoints/{ENDPOINT_NAME}/invocations`
- MLflow Experiment: `{EXPERIMENT_PATH}`
- Model Registry: Unity Catalog → `{CATALOG_NAME}` → `{GOLD_SCHEMA}` → Models